# Constellation multi-model training
Select a GPU runtime, then run all cells. Two patch encoders train on attributed ESO observations and synthetic star fields. Whole scenes are held out for candidate, presence and membership calibration. The run exports an experimental CSV, a recommended CSV, checkpoints and a report. A new model is recommended only if it passes the recorded promotion gate; otherwise the recommended CSV is the scored v4 baseline. No Kaggle score is guaranteed.

In [ ]:
from pathlib import Path
import subprocess, sys
ROOT = Path('/content/constellation-multimodel')
REPO = 'https://github.com/7dracoder/Constellation-Detection---CS-GY-6643.git'
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO, str(ROOT)], check=True)
else:
    changes = subprocess.check_output(['git', '-C', str(ROOT), 'status', '--porcelain'], text=True)
    if changes.strip():
        raise RuntimeError('The existing checkout has changes. Use a different ROOT to preserve them.')
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)
import torch, cv2, scipy, numpy
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('commit:', subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
# ~8,000 total training updates; checkpoints and image downloads are reused on reruns.
command = [sys.executable, '-u', 'multimodel_pipeline.py', '--root', str(ROOT),
           '--download', '--steps', '4000', '--proposals', '12000', '--trials', '3']
subprocess.run(command, cwd=ROOT, check=True)


In [ ]:
import json, zipfile
from google.colab import files
out = ROOT / 'outputs' / 'multimodel'
report = json.loads((out / 'report.json').read_text())
print('Passed promotion:', report['promoted'])
print('Recommended source:', report['recommended_source'])
print('Held-out baseline:', report['baseline']['mean'])
print('Held-out ensemble:', report['challenger']['mean'])
files.download(str(out / 'submission_recommended.csv'))
bundle = ROOT / 'outputs' / 'multimodel_results.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(out.iterdir()):
        if path.suffix in ('.csv', '.json', '.pt') or path.name == 'candidate_classifier.npz':
            archive.write(path, arcname=path.name)
    archive.write(ROOT / 'external' / 'training_images' / 'manifest.json', arcname='external_sources.json')
files.download(str(bundle))
